In [179]:
text = open('input.txt', 'r').read()
text[:10]

'First Citi'

In [180]:
# TODO try words instead of chars
unique_chars = sorted(list(set(text)))

In [181]:
tokenizer_map = {char : i for char,i in enumerate(unique_chars)}
untokenizer_map = {i : char for char,i in enumerate(unique_chars)}
encode = lambda string : [tokenizer_map[char] for char in string]
encode = lambda token_list : [untokenizer_map[token] for token in token_list]

In [182]:
import torch
full_text = torch.tensor(encode(text), dtype=torch.int64) # must be int64

In [183]:
# TODO tune
CONTEXT_WINDOW_SIZE = 32
N_BATCHES = 3
LEARNING_RATE = 1e-3
LEARNING_RATE_DECAY = 0.999
EPOCHS = int(1e5)
N_ATTENTION_HEADS = 4
N_FEATURE_DIMS = 64
QUERY_SIZE = 4
N_UNIQUE_CHARS = 65
SEED = 42
N_HIDDEN_NEURONS = 200

In [184]:
dev_cutoff = int(0.9 * len(full_text)) # TODO add test split
train_data = full_text[:dev_cutoff]
dev_data = full_text[dev_cutoff:]

In [185]:
def GetRandomBatch(data):
    batch_start_indices =  torch.randint(low=0, high=len(data) - CONTEXT_WINDOW_SIZE, size=(N_BATCHES,))
    inputs = torch.stack([data[i:i+CONTEXT_WINDOW_SIZE] for i in batch_start_indices])
    labels = torch.stack([data[i+1:i+CONTEXT_WINDOW_SIZE+1] for i in batch_start_indices])
    return inputs, labels


In [186]:
import math
import copy

class AttentionHead():
    def __init__(self):
        self.query_matrix = torch.randn((N_FEATURE_DIMS, QUERY_SIZE)) # TODO looks backwards
        self.key_matrix = torch.randn((N_FEATURE_DIMS, QUERY_SIZE))
        self.value_matrix_up = torch.randn((N_FEATURE_DIMS, QUERY_SIZE))
        self.value_matrix_down = torch.randn((QUERY_SIZE, N_FEATURE_DIMS))
        self.params = [self.query_matrix, self.key_matrix, self.value_matrix_up, self.value_matrix_down]
        for param in self.params:
            param.requires_grad = True

        self.query_vectors = torch.empty((QUERY_SIZE, CONTEXT_WINDOW_SIZE))
        self.key_vectors = torch.empty((QUERY_SIZE, CONTEXT_WINDOW_SIZE))
        self.attention_matrix = torch.empty((CONTEXT_WINDOW_SIZE, CONTEXT_WINDOW_SIZE))
        self.attended_feature_vectors = torch.empty((CONTEXT_WINDOW_SIZE, N_FEATURE_DIMS))

    def attend(self, feature_vectors):
        self.attended_feature_vectors = copy.deepcopy(feature_vectors)
        query_vectors = feature_vectors @ self.query_matrix  # TODO pytorch vectorize
        key_vectors = feature_vectors @ self.key_matrix # TODO pytorch vectorize
        attention_matrix = query_vectors @ key_vectors.t()
        attention_matrix /= math.sqrt(N_FEATURE_DIMS)
        for row in range(CONTEXT_WINDOW_SIZE):
            for col in range(CONTEXT_WINDOW_SIZE):
                if (row >= col):
                    attention_matrix[row,col] = -math.inf;
        torch.softmax(attention_matrix, dim=0)
        for row in range(CONTEXT_WINDOW_SIZE):
            for col in range(CONTEXT_WINDOW_SIZE):
                # print(self.attended_feature_vectors[row].shape)
                # print(attention_matrix[row,col].shape, self.value_matrix_up.shape, self.value_matrix_down.shape, feature_vectors[row].shape)
                # print(attention_matrix[row,col] * self.value_matrix_up * self.value_matrix_down * feature_vectors[row])
                # print((attention_matrix[row,col] * self.value_matrix_up).shape)
                # print((attention_matrix[row,col] * self.value_matrix_up * self.value_matrix_down).shape)
                # print((attention_matrix[row,col] * self.value_matrix_up * self.value_matrix_down * feature_vectors[row]).shape)
                self.attended_feature_vectors[row] += attention_matrix[row,col] * self.value_matrix_up @ self.value_matrix_down @ feature_vectors[row]
        return self.attended_feature_vectors

In [187]:
class FeedForward():
    def __init__(self):
        g = torch.Generator().manual_seed(SEED) # for reproducibility
        self.hidden_weights = torch.randn((CONTEXT_WINDOW_SIZE * N_FEATURE_DIMS, N_HIDDEN_NEURONS), generator=g)
        self.hidden_biases = torch.randn((N_HIDDEN_NEURONS), generator=g)
        self.output_weights = torch.randn((N_HIDDEN_NEURONS, N_UNIQUE_CHARS), generator=g)
        self.output_biases = torch.randn((N_UNIQUE_CHARS,), generator=g)
        self.params = [self.hidden_weights, self.hidden_biases, self.output_weights, self.output_biases]

    def forward(self, context):
        # context: [BATCH_SIZE, CONTEXT_WINDOW_SIZE, N_FEATURE_DIMS]
        hidden_output = context.view(-1, CONTEXT_WINDOW_SIZE * N_FEATURE_DIMS) @ self.hidden_weights + self.hidden_biases
        output = hidden_output @ self.output_weights + self.output_biases
        return output

In [188]:
import torch.nn.functional as F

def ResetGrad(params):
    for param in params:
        param.grad = None

def ApplyGrad(params, learning_rate):
    for param in params:
        param.data -= learning_rate * param.grad

class Transformer():
    def __init__(self):
        g = torch.Generator().manual_seed(SEED) # for reproducibility
        self.feature_embedding_table = torch.randn((N_UNIQUE_CHARS, N_FEATURE_DIMS))
        self.attention_heads = [AttentionHead() for _ in range(N_ATTENTION_HEADS)]
        self.feed_forward = FeedForward()
        self.params = [self.feature_embedding_table] + self.feed_forward.params 
        for attention_head in self.attention_heads:
            self.params += self.feed_forward.params
        self.learning_rate = LEARNING_RATE

    def forward(self, context):
        print(self.feature_embedding_table.shape, context.shape)
        feature_vectors = self.feature_embedding_table[context]
        print(feature_vectors.shape)
        attended_feature_vectors = torch.concat([attention_head.attend(feature_vectors.view((-1, N_FEATURE_DIMS))) for attention_head in self.attention_heads])
        output = self.feed_forward.forward(attended_feature_vectors)
        return output[0]
    
    def backward(self, output, label):
        loss = F.cross_entropy(output, label)
        ResetGrad(self.params)
        loss.backward()
        ApplyGrad(self.params, self.learning_rate)

    def summarize(self):
        print("=====ATTENTION=====")
        print(f"For each {N_ATTENTION_HEADS} head, query matrix is {self.attention_heads[0].query_matrix.shape}, key matrix is {self.attention_heads[0].key_matrix.shape}, value matrix is {self.attention_heads[0].value_matrix_up.shape} x {self.attention_heads[0].value_matrix_down.shape}")

In [189]:
transformer = Transformer()
X_batch, Y_batch = GetRandomBatch(train_data)

In [190]:
transformer.summarize()

=====ATTENTION=====
For each 4 head, query matrix is torch.Size([64, 4]), key matrix is torch.Size([64, 4]), value matrix is torch.Size([64, 4]) x torch.Size([4, 64])


In [191]:
transformer.forward(X_batch)

torch.Size([65, 64]) torch.Size([3, 32])
torch.Size([3, 32, 64])


tensor([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
       grad_fn=<SelectBackward0>)